In [35]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.tree import DecisionTreeClassifier

In [36]:
# ---------------------------------------------------------------------------
# 1. Carregar
# O arquivo não tem cabeçalho;
# ---------------------------------------------------------------------------
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"
LOCAL = Path("breast-cancer-wisconsin.data")
COLUNAS = [
    "id", "clump_thickness", "uniformity_cell_size", "uniformity_cell_shape",
    "marginal_adhesion", "single_epithelial_cell_size", "bare_nuclei",
    "bland_chromatin", "normal_nucleoli", "mitoses", "class",
]
CLASSES = [2, 4]
NOMES = {2: "benigno (2)", 4: "maligno (4)"}

df_raw = pd.read_csv(LOCAL if LOCAL.exists() else URL, header=None, names=COLUNAS, na_values="?")
df = df_raw.copy()
print(f"Dimensão do dataset: \n{df.shape}\n")
print(f"Tipos de dados: \n{df.dtypes}\n")
print(f"Instâncias com valores ausentes por coluna: \n{df.isna().sum()}\n")
print(f"Imprima o dataset: \n{df.head(10)}\n")

Dimensão do dataset: 
(699, 11)

Tipos de dados: 
id                               int64
clump_thickness                  int64
uniformity_cell_size             int64
uniformity_cell_shape            int64
marginal_adhesion                int64
single_epithelial_cell_size      int64
bare_nuclei                    float64
bland_chromatin                  int64
normal_nucleoli                  int64
mitoses                          int64
class                            int64
dtype: object

Instâncias com valores ausentes por coluna: 
id                              0
clump_thickness                 0
uniformity_cell_size            0
uniformity_cell_shape           0
marginal_adhesion               0
single_epithelial_cell_size     0
bare_nuclei                    16
bland_chromatin                 0
normal_nucleoli                 0
mitoses                         0
class                           0
dtype: int64

Imprima o dataset: 
        id  clump_thickness  uniformity_cell_size  un

In [38]:
# ---------------------------------------------------------------------------
# 2. Limpar
# 'id' não descreve o tumor, só adiciona ruído. Instância com valor ausente
# não pode ser comparada por distância; a tarefa pede descarte.
# ---------------------------------------------------------------------------
df = df_raw.dropna().drop(columns="id").astype(int)
X = df_raw.drop(columns="class").to_numpy()
y = df_raw["class"].to_numpy()
print(f"Quantos dados ausentes no dataset inicial: {df_raw.isna().sum().sum()}\n")
print(f"Instâncias após limpeza: {len(df)}")
print(df["class"].value_counts().rename(NOMES).to_string(), "\n")

Quantos dados ausentes no dataset inicial: 16

Instâncias após limpeza: 683
class
benigno (2)    444
maligno (4)    239 



In [39]:
# ---------------------------------------------------------------------------
# 3. Treino e teste
# O modelo é ajustado no treino; o teste (30%) estima o acerto em dados novos.
# stratify mantém a proporção das classes; random_state torna reproduzível.
# ---------------------------------------------------------------------------
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Treino: {len(X_treino)} | Teste: {len(X_teste)}\n")

Treino: 489 | Teste: 210



In [ ]:
# ---------------------------------------------------------------------------
# 4. Classificador de centróides médios
# Treino: centróide de cada classe = média dos seus exemplos.
# Predição: classe do centróide mais próximo (distância euclidiana).
# Atributos já estão na mesma escala (1-10), então não há padronização.
# ---------------------------------------------------------------------------
class CentroidesMedios:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.centroides_ = np.array([X[y == c].mean(axis=0) for c in self.classes_])
        return self

    def predict(self, X):
        distancias = np.linalg.norm(X[:, None, :] - self.centroides_[None, :, :], axis=2)
        return self.classes_[distancias.argmin(axis=1)]


modelo = CentroidesMedios().fit(X_treino, y_treino)
print("Centróides:")
print(pd.DataFrame(modelo.centroides_, index=[f"centróide {NOMES[c]}" for c in CLASSES],
                   columns=COLUNAS[1:-1]).round(2).to_string(), "\n")